# Drone Weather A2A Demo — AdkApp + sub_agents + Secret Manager + OTel Topology

**Architecture (v3):**
```
User / Gemini Enterprise
  └─▶ Drone Mission Planner  (Agent Engine — AdkApp | Agent Identity SA)
        └─▶ sub_agents: RemoteA2aAgent (ADK-native A2A client)
              └─▶ Weather Agent  (Agent Engine — A2aAgent)
                    ├─▶ get_lat_long() → Google Maps MCP
                    └─▶ get_weather()  → NWS API (real 3-day forecast)
```

**Steps:**
1. Configure project variables
2. Enable required APIs
3. Create GCS staging bucket & grant IAM permissions
3b. Create Maps API key and store in Secret Manager
3c. Grant Secret Manager access to agents
3d. Create Agent Identity (dedicated SA for Mission Planner)
3e. Grant Gemini Enterprise access to Mission Planner
4. Install Python dependencies
5. Deploy Weather Agent (A2A) — reads Maps key from Secret Manager, OTel instrumentation
6. Test Weather Agent directly
7. Deploy Drone Mission Planner (AdkApp + sub_agents + Agent Identity)
8. Run the full demo — via stream_query() and A2A


---
## Step 1 — Configure Project Variables

Update **all four variables** before running any other cell.


In [ ]:
# ── Update these for your environment ──────────────────────────────────────
PROJECT_ID  = "agentcloud-stable-playground"          # GCP project ID
LOCATION    = "us-central1"                   # Vertex AI region
BUCKET_NAME = "drone_staging_bucket_v2"     # New GCS bucket to create (no gs:// prefix)
# Maps API key is created and stored in Secret Manager in Step 3b
# ───────────────────────────────────────────────────────────────────────────

STAGING_BUCKET = f"gs://{BUCKET_NAME}"

import subprocess
subprocess.run(["gcloud", "config", "set", "project", PROJECT_ID], check=True)

PROJECT_NUMBER = subprocess.check_output(
    ["gcloud", "projects", "describe", PROJECT_ID, "--format=value(projectNumber)"]
).decode().strip()

print(f"Project ID    : {PROJECT_ID}")
print(f"Project Number: {PROJECT_NUMBER}")
print(f"Location      : {LOCATION}")
print(f"Staging Bucket: {STAGING_BUCKET}")


Project ID    : agentcloud-stable-playground
Project Number: 100161839600
Location      : us-central1
Staging Bucket: gs://drone_staging_bucket_v2


In [ ]:
# Verify active credentials
!gcloud auth list
!gcloud config get-value project


 Credentialed Accounts
ACTIVE  ACCOUNT
*       ksiri@google.com

To set the active account, run:
    $ gcloud config set account `ACCOUNT`

agentcloud-stable-playground


---
## Step 2 — Enable Required GCP APIs

This step is safe to re-run — already-enabled APIs are skipped.


In [ ]:
REQUIRED_APIS = [
    "aiplatform.googleapis.com",
    "storage.googleapis.com",
    "cloudbuild.googleapis.com",
    "artifactregistry.googleapis.com",
    "iam.googleapis.com",
    "iamcredentials.googleapis.com",
    "compute.googleapis.com",
    "logging.googleapis.com",
    "cloudtrace.googleapis.com",
    "monitoring.googleapis.com",
    "places.googleapis.com",
    "apikeys.googleapis.com",
    "secretmanager.googleapis.com",
]

import subprocess
print("Enabling required APIs...\n")
result = subprocess.run(
    ["gcloud", "services", "enable"] + REQUIRED_APIS + ["--project", PROJECT_ID],
    capture_output=True, text=True
)
if result.returncode == 0:
    for api in REQUIRED_APIS:
        print(f"  ✓ {api}")
    print("\nAll APIs enabled.")
else:
    print(result.stderr)


Enabling required APIs...

  ✓ aiplatform.googleapis.com
  ✓ storage.googleapis.com
  ✓ cloudbuild.googleapis.com
  ✓ artifactregistry.googleapis.com
  ✓ iam.googleapis.com
  ✓ iamcredentials.googleapis.com
  ✓ compute.googleapis.com
  ✓ logging.googleapis.com
  ✓ cloudtrace.googleapis.com
  ✓ monitoring.googleapis.com
  ✓ places.googleapis.com
  ✓ apikeys.googleapis.com
  ✓ secretmanager.googleapis.com

All APIs enabled.


---
## Step 3 — Create GCS Staging Bucket & Grant IAM Permissions

| Principal | Role | Why |
|-----------|------|-----|
| Agent Engine SA | `storage.objectAdmin` on bucket | Stage agent packages |
| Agent Engine SA | `aiplatform.user` on project | Call other Agent Engine A2A endpoints |
| Vertex AI Service Agent | `storage.objectAdmin` on bucket | Cloud Build reads packages |


In [ ]:
import subprocess

print(f"Creating bucket: {STAGING_BUCKET}")
r = subprocess.run(
    ["gcloud", "storage", "buckets", "create", STAGING_BUCKET,
     "--project", PROJECT_ID, "--location", LOCATION, "--uniform-bucket-level-access"],
    capture_output=True, text=True
)
if r.returncode == 0:
    print(f"  ✓ Bucket created")
elif "already exists" in r.stderr:
    print(f"  ✓ Bucket already exists")
else:
    print("Error:", r.stderr)


Creating bucket: gs://drone_staging_bucket_v2
  ✓ Bucket created


In [ ]:
import subprocess

AE_SA  = f"service-{PROJECT_NUMBER}@gcp-sa-aiplatform-re.iam.gserviceaccount.com"
VAI_SA = f"service-{PROJECT_NUMBER}@gcp-sa-aiplatform.iam.gserviceaccount.com"

grants = [
    (["storage", "buckets", "add-iam-policy-binding", STAGING_BUCKET,
      "--member", f"serviceAccount:{AE_SA}", "--role", "roles/storage.objectAdmin"],
     "storage.objectAdmin on bucket → Agent Engine SA"),
    (["storage", "buckets", "add-iam-policy-binding", STAGING_BUCKET,
      "--member", f"serviceAccount:{VAI_SA}", "--role", "roles/storage.objectAdmin"],
     "storage.objectAdmin on bucket → Vertex AI SA"),
    (["projects", "add-iam-policy-binding", PROJECT_ID,
      "--member", f"serviceAccount:{AE_SA}", "--role", "roles/aiplatform.user"],
     "aiplatform.user on project → Agent Engine SA"),
]

for cmd_args, label in grants:
    r = subprocess.run(["gcloud"] + cmd_args, capture_output=True, text=True)
    print(f"  ✓ {label}" if r.returncode == 0 or "already" in r.stderr else f"  ! {label}: {r.stderr[:150]}")

print("\nIAM permissions configured.")


  ✓ storage.objectAdmin on bucket → Agent Engine SA
  ✓ storage.objectAdmin on bucket → Vertex AI SA
  ✓ aiplatform.user on project → Agent Engine SA

IAM permissions configured.


---
## Step 3b — Create Maps API Key and Store in Secret Manager

**Why Secret Manager instead of an env var?**
- API keys in env vars are visible in deployment configs, logs, and container metadata
- Secret Manager stores the key encrypted at rest with versioning and audit logs
- The agent fetches the key at runtime via the metadata server — never baked into the image

This step:
1. Creates a Maps API key restricted to the Places API
2. Creates the `drone-maps-api-key` secret in Secret Manager
3. Stores the key as the first version


In [ ]:
  import subprocess, json, httpx, asyncio, tempfile, os

  # 1. Create Maps API key restricted to Places API
  print("Creating Maps API key restricted to Places API...")
  create_result = subprocess.run(
      ["gcloud", "services", "api-keys", "create",
       "--display-name=Drone Weather Maps Key",
       "--api-target=service=places.googleapis.com",
       "--project", PROJECT_ID, "--billing-project", PROJECT_ID,
       "--format=json", "--quiet"],
      capture_output=True, text=True,
  )

  # gcloud sometimes writes result JSON to stderr for long-running operations
  output = create_result.stdout.strip() or create_result.stderr.strip()
  try:
      result_data = json.loads(output)
      key_name = result_data["name"]
      MAPS_API_KEY = result_data["keyString"]
  except (json.JSONDecodeError, KeyError):
      list_result = subprocess.run(
          ["gcloud", "services", "api-keys", "list",
           "--project", PROJECT_ID, "--billing-project", PROJECT_ID,
           "--filter=displayName='Drone Weather Maps Key'", "--format=json"],
          capture_output=True, text=True,
      )
      keys = json.loads(list_result.stdout or "[]")
      if not keys:
          raise RuntimeError(f"Key creation failed.\n{create_result.stderr}")
      key_name = keys[0]["name"]
      MAPS_API_KEY = subprocess.run(
          ["gcloud", "services", "api-keys", "get-key-string", key_name,
           "--billing-project", PROJECT_ID, "--format=value(keyString)", "--quiet"],
          capture_output=True, text=True, check=True,
      ).stdout.strip()

  print(f"  ✓ Maps API key: {MAPS_API_KEY[:8]}...{MAPS_API_KEY[-4:]}")

  # 2. Create Secret Manager secret
  print("\nCreating Secret Manager secret: drone-maps-api-key...")
  r = subprocess.run(
      ["gcloud", "secrets", "create", "drone-maps-api-key",
       "--project", PROJECT_ID, "--billing-project", PROJECT_ID,
       "--replication-policy=automatic"],
      capture_output=True, text=True,
  )
  if r.returncode == 0:
      print("  ✓ Secret created")
  elif "already exists" in r.stderr:
      print("  ✓ Secret already exists")
  else:
      print(f"  ! {r.stderr[:200]}")

  # 3. Add secret version
  with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f:
      f.write(MAPS_API_KEY)
      tmp_path = f.name

  r = subprocess.run(
      ["gcloud", "secrets", "versions", "add", "drone-maps-api-key",
       "--data-file", tmp_path,
       "--project", PROJECT_ID, "--billing-project", PROJECT_ID],
      capture_output=True, text=True,
  )
  os.unlink(tmp_path)
  print("  ✓ Secret version added" if r.returncode == 0 else f"  ! {r.stderr[:200]}")

  print(f"\nMaps API key stored in Secret Manager as: projects/{PROJECT_ID}/secrets/drone-maps-api-key")

Creating Maps API key restricted to Places API...
  ✓ Maps API key: AIzaSyCM...jCZ4

Creating Secret Manager secret: drone-maps-api-key...
  ✓ Secret created
  ✓ Secret version added

Maps API key stored in Secret Manager as: projects/agentcloud-stable-playground/secrets/drone-maps-api-key


In [ ]:
  import subprocess, json, httpx, asyncio
  from google.cloud import secretmanager

  def _get_maps_api_key() -> str:
      client = secretmanager.SecretManagerServiceClient()
      name = f"projects/{PROJECT_ID}/secrets/drone-maps-api-key/versions/latest"
      return client.access_secret_version(name=name).payload.data.decode("utf-8")

  MAPS_API_KEY = _get_maps_api_key()
  print(f"  ✓ Fetched Maps API key: {MAPS_API_KEY[:8]}...{MAPS_API_KEY[-4:]}")

  async def verify_maps_key():
      async with httpx.AsyncClient(timeout=10) as client:
          resp = await client.post(
              "https://mapstools.googleapis.com/mcp",
              json={"jsonrpc": "2.0", "id": 1, "method": "initialize",
                    "params": {"protocolVersion": "2024-11-05",
                                "clientInfo": {"name": "test", "version": "1.0"}, "capabilities": {}}},
              headers={"X-Goog-Api-Key": MAPS_API_KEY, "Content-Type": "application/json"},
          )
      print("✓ Maps API key verified" if resp.status_code == 200 else f"✗ HTTP {resp.status_code}: {resp.text[:200]}")

  await verify_maps_key()

  ✓ Fetched Maps API key: AIzaSyCM...jCZ4
✓ Maps API key verified


---
## Step 3c — Grant Secret Manager Access to Agents

Both the Weather Agent and the Mission Planner need `secretmanager.secretAccessor` to read the Maps API key at runtime.

| Principal | Role | Why |
|-----------|------|-----|
| Agent Engine RE SA | `secretmanager.secretAccessor` | Weather Agent reads key at runtime |
| `drone-mission-planner@` SA | `secretmanager.secretAccessor` | Planner SA (created in 3d) also needs access |


In [ ]:
import subprocess

AE_SA = f"service-{PROJECT_NUMBER}@gcp-sa-aiplatform-re.iam.gserviceaccount.com"
PLANNER_SA = f"drone-mission-planner@{PROJECT_ID}.iam.gserviceaccount.com"
SECRET_NAME = f"projects/{PROJECT_ID}/secrets/drone-maps-api-key"

for sa, label in [(AE_SA, "Agent Engine RE SA"), (PLANNER_SA, "drone-mission-planner SA")]:
    r = subprocess.run(
        ["gcloud", "secrets", "add-iam-policy-binding", "drone-maps-api-key",
         "--project", PROJECT_ID,
         "--member", f"serviceAccount:{sa}",
         "--role", "roles/secretmanager.secretAccessor"],
        capture_output=True, text=True
    )
    print(f"  ✓ secretmanager.secretAccessor granted to {label}" if r.returncode == 0 or "already" in r.stderr
          else f"  ! {label}: {r.stderr[:150]}")

print("\nSecret Manager access configured.")


  ✓ secretmanager.secretAccessor granted to Agent Engine RE SA
  ✓ secretmanager.secretAccessor granted to drone-mission-planner SA

Secret Manager access configured.


---
## Step 3d — Create Agent Identity (Dedicated SA for Mission Planner)

**Why Agent Identity?**

In the default setup all Agent Engine agents share one platform SA. With Agent Identity each agent gets its own:

| Feature | Shared Platform SA | Agent Identity |
|---|---|---|
| Principal in logs | Generic `gcp-sa-aiplatform-re` | `drone-mission-planner@project.iam.gserviceaccount.com` |
| IAM scope | Broad `aiplatform.user` project-wide | Least-privilege — only what this agent needs |
| Token binding | Bearer token only | mTLS-bound to Agent Engine runtime |
| Blast radius | Any agent breach = full access | Breach limited to this agent's permissions |

**This step creates the SA, grants least-privilege roles, and allows Agent Engine to impersonate it.**


In [ ]:
import subprocess

PLANNER_SA = f"drone-mission-planner@{PROJECT_ID}.iam.gserviceaccount.com"
AE_SA = f"service-{PROJECT_NUMBER}@gcp-sa-aiplatform-re.iam.gserviceaccount.com"

print(f"Creating dedicated SA: {PLANNER_SA}\n")

# 1. Create SA
r = subprocess.run(
    ["gcloud", "iam", "service-accounts", "create", "drone-mission-planner",
     "--project", PROJECT_ID,
     "--display-name", "Drone Mission Planner Agent Identity",
     "--description", "Unique identity for the Drone Mission Planner"],
    capture_output=True, text=True,
)
print("  ✓ SA created" if r.returncode == 0 else f"  ✓ SA already exists")

# 2. Grant least-privilege roles
for role in ["roles/aiplatform.user", "roles/logging.logWriter", "roles/cloudtrace.agent",
             "roles/secretmanager.secretAccessor"]:
    r = subprocess.run(
        ["gcloud", "projects", "add-iam-policy-binding", PROJECT_ID,
         "--member", f"serviceAccount:{PLANNER_SA}", "--role", role],
        capture_output=True, text=True,
    )
    print(f"  ✓ Granted {role}")

# 3. Allow Agent Engine SA to impersonate planner SA
r = subprocess.run(
    ["gcloud", "iam", "service-accounts", "add-iam-policy-binding", PLANNER_SA,
     "--project", PROJECT_ID,
     "--member", f"serviceAccount:{AE_SA}",
     "--role", "roles/iam.serviceAccountTokenCreator"],
    capture_output=True, text=True,
)
print("  ✓ Granted serviceAccountTokenCreator to Agent Engine SA")

AE_SA = "service-100161839600@gcp-sa-aiplatform-re.iam.gserviceaccount.com"

r = subprocess.run([
      "gcloud", "projects", "add-iam-policy-binding", PROJECT_ID,
      "--member", f"serviceAccount:{AE_SA}",
      "--role", "roles/mcp.toolUser",
      "--billing-project", PROJECT_ID,
  ], check=True)

print(f"\nGranted mcp Tool user access to: {AE_SA}")


Creating dedicated SA: drone-mission-planner@agentcloud-stable-playground.iam.gserviceaccount.com

  ✓ SA already exists
  ✓ Granted roles/aiplatform.user
  ✓ Granted roles/logging.logWriter
  ✓ Granted roles/cloudtrace.agent
  ✓ Granted roles/secretmanager.secretAccessor
  ✓ Granted serviceAccountTokenCreator to Agent Engine SA

Agent Identity ready: drone-mission-planner@agentcloud-stable-playground.iam.gserviceaccount.com


---
## Step 3e — Grant Gemini Enterprise Access to Mission Planner

Gemini Enterprise calls Agent Engine agents via the `cloudaicompanion` service account.  
For the Planner to run as its Agent Identity (`drone-mission-planner@`) when invoked from GE,  
the `cloudaicompanion` SA needs two grants:

| Principal | Role | Why |
|-----------|------|-----|
| `cloudaicompanion` SA | `aiplatform.user` on project | Invoke Agent Engine agents |
| `cloudaicompanion` SA | `serviceAccountTokenCreator` on `drone-mission-planner@` | Impersonate the Agent Identity |

> **Note:** Without `serviceAccountTokenCreator`, Gemini Enterprise receives a 403 when trying  
> to invoke the Mission Planner with its Agent Identity. The 403 surfaces as a 404 at the GE layer.


In [ ]:
import subprocess

GE_SA = f"service-{PROJECT_NUMBER}@gcp-sa-cloudaicompanion.iam.gserviceaccount.com"
PLANNER_SA = f"drone-mission-planner@{PROJECT_ID}.iam.gserviceaccount.com"

print(f"Configuring Gemini Enterprise access...")
print(f"  GE SA    : {GE_SA}")
print(f"  Planner SA: {PLANNER_SA}\n")

# 1. aiplatform.user on project — allows GE SA to invoke Agent Engine
r = subprocess.run(
    ["gcloud", "projects", "add-iam-policy-binding", PROJECT_ID,
     "--member", f"serviceAccount:{GE_SA}",
     "--role", "roles/aiplatform.user"],
    capture_output=True, text=True
)
print("  ✓ aiplatform.user granted to cloudaicompanion SA" if r.returncode == 0 or "already" in r.stderr
      else f"  ! {r.stderr[:200]}")

# 2. serviceAccountTokenCreator on planner SA — allows GE SA to impersonate Agent Identity
r = subprocess.run(
    ["gcloud", "iam", "service-accounts", "add-iam-policy-binding", PLANNER_SA,
     "--project", PROJECT_ID,
     "--member", f"serviceAccount:{GE_SA}",
     "--role", "roles/iam.serviceAccountTokenCreator"],
    capture_output=True, text=True
)
print("  ✓ serviceAccountTokenCreator granted to cloudaicompanion SA on planner SA" if r.returncode == 0 or "already" in r.stderr
      else f"  ! {r.stderr[:200]}")

print(f"\nGemini Enterprise can now invoke the Mission Planner as {PLANNER_SA}")


Configuring Gemini Enterprise access...
  GE SA    : service-100161839600@gcp-sa-cloudaicompanion.iam.gserviceaccount.com
  Planner SA: drone-mission-planner@agentcloud-stable-playground.iam.gserviceaccount.com

  ✓ aiplatform.user granted to cloudaicompanion SA
  ✓ serviceAccountTokenCreator granted to cloudaicompanion SA on planner SA

Gemini Enterprise can now invoke the Mission Planner as drone-mission-planner@agentcloud-stable-playground.iam.gserviceaccount.com


---
## Step 4 — Install Python Dependencies


In [ ]:
!pip install --upgrade pip setuptools wheel
!pip install -q --upgrade \
    "google-cloud-aiplatform>=1.148.0" \
    "google-adk[a2a]==1.30.0" \
    "a2a-sdk==0.3.26" \
    "fastmcp>=2.0.0" \
    "httpx" \
    "requests" \
    "google-auth>=2.29.0" \
    "google-cloud-secret-manager"

import subprocess
result = subprocess.check_output(["pip", "show", "google-auth"]).decode()
version_line = [l for l in result.splitlines() if l.startswith("Version")][0]
print(f"Installed: {version_line}")

#print("Restarting kernel to apply upgrades...")
#import IPython
#IPython.Application.instance().kernel.do_shutdown(restart=True)


Installed: Version: 2.49.2


---
## Step 5 — Deploy Weather Agent (A2A)

The Weather Agent is deployed as `A2aAgent` (pure A2A interface).  
**Changes from v2:**
- Maps API key is read from **Secret Manager** at runtime via `_get_maps_api_key()` — not from env var
- **OTel instrumentation packages** added: `opentelemetry-instrumentation-google-genai`, `opentelemetry-instrumentation-httpx`, `opentelemetry-instrumentation-vertexai`
- **OTel env vars** added to emit telemetry to GCS for Agent Engine topology visibility

> **Note:** Deployment takes 3–5 minutes.


In [ ]:
import os
import vertexai
from vertexai import agent_engines
from vertexai.preview.reasoning_engines import A2aAgent
from a2a.types import AgentCard, AgentCapabilities, AgentSkill


def _get_maps_api_key() -> str:
    """Read Maps API key from Secret Manager at runtime."""
    from google.cloud import secretmanager
    client = secretmanager.SecretManagerServiceClient()
    response = client.access_secret_version(
        request={"name": f"projects/{os.environ.get('GOOGLE_CLOUD_PROJECT', '')}/secrets/drone-maps-api-key/versions/latest"}
    )
    return response.payload.data.decode("UTF-8")


async def get_lat_long(city: str) -> dict:
    """Use Google Maps MCP to get latitude and longitude for a city."""
    from fastmcp import Client
    from fastmcp.client.transports import StreamableHttpTransport
    maps_api_key = _get_maps_api_key()
    transport = StreamableHttpTransport(
        url="https://mapstools.googleapis.com/mcp",
        headers={"X-Goog-Api-Key": maps_api_key},
    )
    async with Client(transport) as client:
        result = await client.call_tool("search_places", {"text_query": city})
        places = (result.structured_content or {}).get("places", [])
        if places:
            loc = places[0].get("location", {})
            return {"latitude": loc.get("latitude"), "longitude": loc.get("longitude")}
        return {"latitude": None, "longitude": None}


def get_weather(latitude: float, longitude: float) -> str:
    """Fetch 3-day forecast from the National Weather Service API."""
    import requests
    headers = {"User-Agent": "DroneWeatherA2A/1.0"}
    pts = requests.get(
        f"https://api.weather.gov/points/{latitude},{longitude}",
        headers=headers, timeout=10,
    )
    forecast_url = pts.json()["properties"]["forecast"]
    forecast = requests.get(forecast_url, headers=headers, timeout=10)
    periods = forecast.json()["properties"]["periods"]
    return "\n".join([
        f"{p['name']}: {p['temperature']}\u00b0{p['temperatureUnit']}, "
        f"{p['shortForecast']}, Wind: {p['windSpeed']} {p['windDirection']}"
        for p in periods[:3]
    ])


def build_weather_executor():
    import os
    from google.adk.agents import Agent
    from google.adk.models import Gemini
    from google.adk.runners import Runner
    from google.adk.sessions import VertexAiSessionService
    from google.genai import types as genai_types
    from a2a.server.agent_execution import AgentExecutor, RequestContext
    from a2a.server.events import EventQueue
    from a2a.server.tasks import TaskUpdater
    from a2a.types import Part, TextPart, UnsupportedOperationError
    from a2a.utils.errors import ServerError

    app_name = os.environ.get("GOOGLE_CLOUD_AGENT_ENGINE_ID", "WeatherAssistant")
    _sessions = VertexAiSessionService()
    _runner = Runner(
        app_name=app_name,
        agent=Agent(
            name="WeatherAssistant",
            model=Gemini(model="gemini-2.5-flash"),
            instruction=(
                "You are a flight-safety weather assistant for drone operators.\n"
                "1. Use get_lat_long to find the city coordinates.\n"
                "2. Use get_weather to fetch the 3-day forecast.\n"
                "3. Analyse each period wind speed carefully:\n"
                "   - winds > 15mph: say GROUNDED — too risky for drone flight.\n"
                "   - winds 10-15mph: say CAUTION — flyable but challenging.\n"
                "   - winds < 10mph: say SAFE — good conditions.\n"
                "4. Report the forecast for all 3 periods and give a final overall recommendation."
            ),
            tools=[get_lat_long, get_weather],
        ),
        session_service=_sessions,
    )

    class WeatherAgentExecutor(AgentExecutor):
        async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
            updater = TaskUpdater(event_queue, context.task_id, context.context_id)
            await updater.start_work()
            query = context.get_user_input()
            session_id = context.context_id
            try:
                await _sessions.create_session(app_name=app_name, user_id="a2a-user", session_id=session_id)
            except Exception:
                pass
            user_msg = genai_types.Content(role="user", parts=[genai_types.Part(text=query)])
            try:
                full_text = ""
                for event in _runner.run(user_id="a2a-user", session_id=session_id, new_message=user_msg):
                    if hasattr(event, "is_final_response") and event.is_final_response():
                        if hasattr(event, "content") and event.content:
                            for p in event.content.parts:
                                if hasattr(p, "text") and p.text:
                                    full_text = p.text
                                    break
                        break
                    if hasattr(event, "content") and event.content:
                        for p in event.content.parts:
                            if hasattr(p, "text") and p.text:
                                full_text = p.text
                                break
                reply = updater.new_agent_message(parts=[Part(root=TextPart(text=full_text or "No response."))])
                await updater.complete(message=reply)
            except Exception as e:
                err = updater.new_agent_message(parts=[Part(root=TextPart(text=f"Error: {e}"))])
                await updater.failed(message=err)

        async def cancel(self, context, event_queue):
            raise ServerError(UnsupportedOperationError())

    return WeatherAgentExecutor()


weather_agent_card = AgentCard(
    name="WeatherAssistant", version="1.0.0", protocolVersion="0.3.0",
    preferredTransport="HTTP+JSON",
    description="Drone weather safety agent — real forecasts via Google Maps MCP + NWS.",
    defaultInputModes=["TEXT"], defaultOutputModes=["TEXT"],
    url="http://placeholder.url",
    capabilities=AgentCapabilities(streaming=False),
    skills=[AgentSkill(id="get_weather_forecast", name="Detailed Weather Forecast",
                       description="3-day forecasts for US cities via Google Maps and NWS.",
                       tags=["Weather", "Forecasting", "DroneSafety"])],
)
print("Weather Agent code defined.")


Weather Agent code defined.


In [ ]:
vertexai.init(project=PROJECT_ID, location=LOCATION, staging_bucket=STAGING_BUCKET)

weather_app = A2aAgent(agent_card=weather_agent_card, agent_executor_builder=build_weather_executor)

print("Deploying Weather Agent — takes 3-5 minutes...")
weather_remote = agent_engines.create(
    weather_app,
    display_name="Weather_Drone_A2A_v3_AR",
    requirements=[
        "google-adk[a2a]==1.30.0",
        "a2a-sdk==0.3.26",
        "fastmcp>=2.0.0",
        "httpx",
        "requests",
        "google-cloud-secret-manager",
        "opentelemetry-instrumentation-google-genai",
        "opentelemetry-instrumentation-httpx",
        "opentelemetry-instrumentation-vertexai",
        "setuptools",
        "pydantic>=2.0.0",
    ],
    env_vars={
        "GOOGLE_GENAI_USE_VERTEXAI": "1",
        "GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY": "true",
        "OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT": "SPAN_AND_EVENT",
        "OTEL_INSTRUMENTATION_GENAI_UPLOAD_FORMAT": "jsonl",
        "OTEL_INSTRUMENTATION_GENAI_COMPLETION_HOOK": "upload",
        "OTEL_SEMCONV_STABILITY_OPT_IN": "gen_ai_latest_experimental",
        "OTEL_INSTRUMENTATION_GENAI_UPLOAD_BASE_PATH": f"gs://{BUCKET_NAME}",
    },
)

WEATHER_AGENT_RESOURCE = weather_remote.resource_name
print(f"\nWeather Agent deployed!")
print(f"Resource: {WEATHER_AGENT_RESOURCE}")
print(f"\n*** Save this resource name ***")


INFO:vertexai.agent_engines:Identified the following requirements: {'pydantic': '2.12.3', 'cloudpickle': '3.1.2', 'google-cloud-aiplatform': '1.148.0', 'a2a-sdk': '0.3.26'}
INFO:vertexai.agent_engines:The following requirements are appended: {'cloudpickle==3.1.2'}
INFO:vertexai.agent_engines:The final list of requirements: ['google-adk[a2a]==1.30.0', 'a2a-sdk==0.3.26', 'fastmcp>=2.0.0', 'httpx', 'requests', 'google-cloud-secret-manager', 'opentelemetry-instrumentation-google-genai', 'opentelemetry-instrumentation-httpx', 'opentelemetry-instrumentation-vertexai', 'setuptools', 'pydantic>=2.0.0', 'cloudpickle==3.1.2']
INFO:vertexai.agent_engines:Using bucket drone_staging_bucket_v2


Deploying Weather Agent — takes 3-5 minutes...


INFO:vertexai.agent_engines:Wrote to gs://drone_staging_bucket_v2/agent_engine/agent_engine.pkl
INFO:vertexai.agent_engines:Writing to gs://drone_staging_bucket_v2/agent_engine/requirements.txt
INFO:vertexai.agent_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.agent_engines:Writing to gs://drone_staging_bucket_v2/agent_engine/dependencies.tar.gz
INFO:vertexai.agent_engines:Creating AgentEngine
INFO:vertexai.agent_engines:Create AgentEngine backing LRO: projects/100161839600/locations/us-central1/reasoningEngines/8966078069617983488/operations/6887110119335657472
INFO:vertexai.agent_engines:View progress and logs at https://console.cloud.google.com/logs/query?project=agentcloud-stable-playground
INFO:vertexai.agent_engines:AgentEngine created. Resource name: projects/100161839600/locations/us-central1/reasoningEngines/8966078069617983488
INFO:vertexai.agent_engines:To use this AgentEngine in another session:
INFO:vertexai.agent_engines:agent_engine = vertexai.agent_e


Weather Agent deployed!
Resource: projects/100161839600/locations/us-central1/reasoningEngines/8966078069617983488

*** Save this resource name ***


---
## Step 6 — Test Weather Agent Directly

Verifies the Weather Agent works end-to-end (Secret Manager key read + Maps MCP + NWS) before wiring up the Mission Planner.


In [ ]:
  import subprocess
  result = subprocess.run([
      "gcloud", "services", "api-keys", "list",
      "--project", PROJECT_ID, "--billing-project", PROJECT_ID,
      "--filter=displayName='Drone Weather Maps Key'",
      "--format=json",
  ], capture_output=True, text=True)
  import json
  keys = json.loads(result.stdout)
  print(json.dumps(keys[0].get("restrictions", {}), indent=2))

  #If it shows places.googleapis.com, update it:

  key_name = keys[0]["name"]
  subprocess.run([
      "gcloud", "services", "api-keys", "update", key_name,
      "--api-target=service=mapstools.googleapis.com",
      "--project", PROJECT_ID, "--billing-project", PROJECT_ID, "--quiet",
  ], check=True)
  print("✓ Key restriction updated to mapstools.googleapis.com")

{
  "apiTargets": [
    {
      "service": "places.googleapis.com"
    }
  ]
}
✓ Key restriction updated to mapstools.googleapis.com


In [ ]:
import asyncio, subprocess, uuid, httpx
AE_SA = "service-100161839600@gcp-sa-aiplatform-re.iam.gserviceaccount.com"

subprocess.run([
      "gcloud", "projects", "add-iam-policy-binding", PROJECT_ID,
      "--member", f"serviceAccount:{AE_SA}",
      "--role", "roles/mcp.toolUser",
      "--billing-project", PROJECT_ID,
  ], check=True)

CompletedProcess(args=['gcloud', 'projects', 'add-iam-policy-binding', 'agentcloud-stable-playground', '--member', 'serviceAccount:service-100161839600@gcp-sa-aiplatform-re.iam.gserviceaccount.com', '--role', 'roles/mcp.toolUser', '--billing-project', 'agentcloud-stable-playground'], returncode=0)

In [ ]:
import asyncio, subprocess, uuid, httpx

_parts = WEATHER_AGENT_RESOURCE.split("/")
WEATHER_A2A_SEND_URL = (
    f"https://{_parts[3]}-aiplatform.googleapis.com/v1beta1/"
    f"{WEATHER_AGENT_RESOURCE}/a2a/v1/message:send"
)
print(f"Weather Agent A2A URL:\n  {WEATHER_A2A_SEND_URL}\n")

def get_token():
    return subprocess.check_output(["gcloud", "auth", "print-access-token"]).decode().strip()

async def call_weather_agent(city: str) -> str:
    payload = {
        "message": {"messageId": str(uuid.uuid4()), "role": "ROLE_USER",
                    "content": [{"text": f"Is it safe to fly a drone in {city}?"}]},
        "configuration": {"blocking": True},
    }
    async with httpx.AsyncClient(timeout=120.0) as client:
        resp = await client.post(
            WEATHER_A2A_SEND_URL, json=payload,
            headers={"Authorization": f"Bearer {get_token()}", "Content-Type": "application/json"},
        )
        resp.raise_for_status()
    return resp.json()["task"]["status"]["message"]["content"][0]["text"]

print("Testing Weather Agent via A2A...\n")
result = await call_weather_agent("Austin, Texas")
print(result)


Weather Agent A2A URL:
  https://us-central1-aiplatform.googleapis.com/v1beta1/projects/100161839600/locations/us-central1/reasoningEngines/8966078069617983488/a2a/v1/message:send

Testing Weather Agent via A2A...

Here's the 3-day forecast for Austin, Texas:

*   **Today:** 90°F, Partly Sunny, Wind: 10 to 15 mph S. **CAUTION** — flyable but challenging.
*   **Tonight:** 67°F, Mostly Cloudy, Wind: 5 to 15 mph S. **CAUTION** — flyable but challenging.
*   **Saturday:** 74°F, Showers And Thunderstorms Likely, Wind: 5 to 15 mph NNW. **CAUTION** — flyable but challenging.

**Overall Recommendation: CAUTION** — while flying is possible, be aware of potentially challenging wind conditions throughout the 3-day forecast.


---
## Step 7 — Deploy Drone Mission Planner (AdkApp + sub_agents + Agent Identity)

**Why AdkApp instead of A2aAgent?**

`A2aAgent` only exposes A2A endpoints. `AdkApp` exposes **both**:
- `/api/stream_reasoning_engine` — used by Gemini Enterprise and `stream_query()`
- `/a2a/...` — used by other A2A agents

**Why `sub_agents` instead of `tools=[AgentTool(...)]`?**

ADK emits agent-to-agent OTel spans only when the weather agent is registered as a `sub_agents` entry.  
Using `AgentTool` wraps it as a tool call — no topology edge appears in Agent Engine observability.

**Auth design inside the planner container:**
```
RemoteA2aAgent.send_message()
  → _make_auth_transport() creates fresh httpx.AsyncClient(auth=_GcpAuth())
  → _GcpAuth._ensure_valid() lazy-inits credentials from metadata server as drone-mission-planner@
  → POST {weather_a2a_url}/v1/message:send
```

**Why lazy credential init + fresh client per call?**
- `AdkApp` cloudpickles the agent at deploy time — `httpx.AsyncClient` (contains `RLock`) can't be deep-copied
- `_GcpAuth.__init__` stores no credentials — only sets `None` — so nothing breaks during cloudpickle
- Fresh client per call via `_make_auth_transport` avoids *Event loop is closed* across Agent Engine workers

> **Note:** Deployment takes 3–5 minutes.


In [ ]:
import os
import vertexai
from vertexai import agent_engines
from vertexai.preview.reasoning_engines import AdkApp
from google.adk.agents import Agent
from google.adk.agents.remote_a2a_agent import RemoteA2aAgent
from google.adk.models import Gemini
import httpx as _httpx
from a2a.client import ClientFactory, ClientConfig
from a2a.client.transports import RestTransport
from a2a.types import AgentCard, AgentCapabilities, AgentSkill, TransportProtocol


# ---------------------------------------------------------------------------
# GCP Auth — lazy init so credentials are never baked into the cloudpickle.
# Credentials are obtained from the metadata server on the first request
# inside the Agent Engine container.
# ---------------------------------------------------------------------------

class _GcpAuth(_httpx.Auth):
    def __init__(self):
        self._credentials = None
        self._transport_request = None

    def _ensure_valid(self):
        import google.auth
        import google.auth.transport.requests as _greq
        if self._credentials is None:
            self._credentials, _ = google.auth.default(
                scopes=["https://www.googleapis.com/auth/cloud-platform"]
            )
            self._transport_request = _greq.Request()
        if not self._credentials.valid:
            self._credentials.refresh(self._transport_request)

    def auth_flow(self, request):
        self._ensure_valid()
        request.headers["Authorization"] = f"Bearer {self._credentials.token}"
        yield request


def _make_auth_transport(card, url, config, interceptors):
    """Fresh httpx.AsyncClient per A2A call — avoids RLock deepcopy and event loop issues."""
    return RestTransport(
        _httpx.AsyncClient(
            timeout=120,
            headers={"Content-Type": "application/json"},
            auth=_GcpAuth(),
        ),
        card, url, interceptors, None,
    )


_authenticated_factory = ClientFactory(
    ClientConfig(
        supported_transports=[TransportProtocol.http_json],
        use_client_preference=True,
    )
)
_authenticated_factory.register(TransportProtocol.http_json, _make_auth_transport)


# ---------------------------------------------------------------------------
# RemoteA2aAgent — ADK-native A2A client for the Weather Agent.
# Agent Engine does not expose /.well-known/agent-card.json so the card
# is constructed directly here.
# ---------------------------------------------------------------------------

def _build_weather_agent(weather_resource: str) -> RemoteA2aAgent:
    resource_parts = weather_resource.split("/")
    location = resource_parts[3]
    weather_a2a_url = (
        f"https://{location}-aiplatform.googleapis.com/v1beta1/{weather_resource}/a2a"
    )

    weather_card = AgentCard(
        name="WeatherAssistant", version="1.0.0", protocolVersion="0.3.0",
        preferredTransport="HTTP+JSON",
        description="Drone weather safety agent — real forecasts via Google Maps MCP + NWS.",
        defaultInputModes=["TEXT"], defaultOutputModes=["TEXT"],
        url=weather_a2a_url,
        capabilities=AgentCapabilities(streaming=False),
        skills=[AgentSkill(
            id="get_weather_forecast", name="Detailed Weather Forecast",
            description="3-day forecasts for US cities using Google Maps and NWS.",
            tags=["Weather", "Forecasting", "DroneSafety"],
        )],
    )

    return RemoteA2aAgent(
        name="WeatherAssistant",
        description=(
            "Drone weather safety agent. Checks real-time conditions for a city "
            "and returns a 3-day forecast with SAFE, CAUTION, or GROUNDED status."
        ),
        agent_card=weather_card,
        a2a_client_factory=_authenticated_factory,
    )


planner_agent = Agent(
    name="DroneMissionPlanner",
    model=Gemini(model="gemini-2.5-flash"),
    instruction=(
        "You are a drone mission planner. When asked to plan or approve a "
        "drone mission, always delegate to WeatherAssistant first to check "
        "weather conditions for the relevant city. Include the user's full "
        "original request when calling WeatherAssistant so the weather agent "
        "has complete context. "
        "If the status is SAFE — approve the mission with details. "
        "If CAUTION — approve with a warning about challenging conditions. "
        "If GROUNDED — reject it and explain the weather conditions."
    ),
    sub_agents=[_build_weather_agent(WEATHER_AGENT_RESOURCE)],
)

print("Drone Mission Planner code defined.")
print(f"  Framework: AdkApp (adk-app)")
print(f"  Sub-agent: WeatherAssistant via RemoteA2aAgent")
print(f"  Weather Agent: {WEATHER_AGENT_RESOURCE}")


Drone Mission Planner code defined.
  Framework: AdkApp (adk-app)
  Sub-agent: WeatherAssistant via RemoteA2aAgent
  Weather Agent: projects/100161839600/locations/us-central1/reasoningEngines/8966078069617983488


/tmp/ipykernel_154/2261950703.py:90: UserWarning: [EXPERIMENTAL] RemoteA2aAgent: ADK Implementation for A2A support (A2aAgentExecutor, RemoteA2aAgent and corresponding supporting components etc.) is in experimental mode and is subject to breaking changes. A2A protocol and SDK are themselves not experimental. Once it's stable enough the experimental mode will be removed. Your feedback is welcome.
  return RemoteA2aAgent(


In [ ]:
vertexai.init(project=PROJECT_ID, location=LOCATION, staging_bucket=STAGING_BUCKET)

app = AdkApp(agent=planner_agent)

print("Deploying Drone Mission Planner (AdkApp + sub_agents + Agent Identity) — takes 3-5 minutes...")
planner_remote = agent_engines.create(
    app,
    display_name="Drone_Mission_Planner_v3_AR",
    requirements=[
        "google-adk[a2a]==1.30.0",
        "a2a-sdk==0.3.26",
        "google-auth>=2.29.0",
        "opentelemetry-instrumentation-google-genai",
        "opentelemetry-instrumentation-httpx",
        "opentelemetry-instrumentation-vertexai",
        "setuptools",
        "pydantic>=2.0.0",
    ],
    env_vars={
        "WEATHER_AGENT_RESOURCE": WEATHER_AGENT_RESOURCE,
        "GOOGLE_GENAI_USE_VERTEXAI": "1",
        "GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY": "true",
        "OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT": "SPAN_AND_EVENT",
        "OTEL_INSTRUMENTATION_GENAI_UPLOAD_FORMAT": "jsonl",
        "OTEL_INSTRUMENTATION_GENAI_COMPLETION_HOOK": "upload",
        "OTEL_SEMCONV_STABILITY_OPT_IN": "gen_ai_latest_experimental",
        "OTEL_INSTRUMENTATION_GENAI_UPLOAD_BASE_PATH": f"gs://{BUCKET_NAME}",
    },
    service_account=PLANNER_SA,
)

PLANNER_RESOURCE = planner_remote.resource_name
print(f"\nDrone Mission Planner deployed!")
print(f"Resource      : {PLANNER_RESOURCE}")
print(f"Agent Identity: {PLANNER_SA}")
print(f"\n*** Save this resource name ***")


INFO:vertexai.agent_engines:Identified the following requirements: {'pydantic': '2.12.3', 'cloudpickle': '3.1.2', 'google-cloud-aiplatform': '1.148.0'}
INFO:vertexai.agent_engines:The following requirements are appended: {'cloudpickle==3.1.2'}
INFO:vertexai.agent_engines:The final list of requirements: ['google-adk[a2a]==1.30.0', 'a2a-sdk==0.3.26', 'google-auth>=2.29.0', 'opentelemetry-instrumentation-google-genai', 'opentelemetry-instrumentation-httpx', 'opentelemetry-instrumentation-vertexai', 'setuptools', 'pydantic>=2.0.0', 'cloudpickle==3.1.2']
INFO:vertexai.agent_engines:Using bucket drone_staging_bucket_v2


Deploying Drone Mission Planner (AdkApp + sub_agents + Agent Identity) — takes 3-5 minutes...


INFO:vertexai.agent_engines:Wrote to gs://drone_staging_bucket_v2/agent_engine/agent_engine.pkl
INFO:vertexai.agent_engines:Writing to gs://drone_staging_bucket_v2/agent_engine/requirements.txt
INFO:vertexai.agent_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.agent_engines:Writing to gs://drone_staging_bucket_v2/agent_engine/dependencies.tar.gz
INFO:vertexai.agent_engines:Creating AgentEngine
INFO:vertexai.agent_engines:Create AgentEngine backing LRO: projects/100161839600/locations/us-central1/reasoningEngines/7384188700504096768/operations/4345162381219856384
INFO:vertexai.agent_engines:View progress and logs at https://console.cloud.google.com/logs/query?project=agentcloud-stable-playground
INFO:vertexai.agent_engines:AgentEngine created. Resource name: projects/100161839600/locations/us-central1/reasoningEngines/7384188700504096768
INFO:vertexai.agent_engines:To use this AgentEngine in another session:
INFO:vertexai.agent_engines:agent_engine = vertexai.agent_e


Drone Mission Planner deployed!
Resource      : projects/100161839600/locations/us-central1/reasoningEngines/7384188700504096768
Agent Identity: drone-mission-planner@agentcloud-stable-playground.iam.gserviceaccount.com

*** Save this resource name ***


---
## Step 8 — Run the Full Demo

**Full request flow:**
```
stream_query() / Gemini Enterprise
  → AdkApp /api/stream_reasoning_engine
  → DroneMissionPlanner (runs as drone-mission-planner@)
  → sub_agents: RemoteA2aAgent → _make_auth_transport() → fresh httpx.AsyncClient(auth=_GcpAuth())
  → POST Weather Agent A2A endpoint
  → WeatherAssistant → _get_maps_api_key() [Secret Manager] → get_lat_long() [Maps MCP] → get_weather() [NWS]
  → SAFE / CAUTION / GROUNDED per period
  → Planner returns go/no-go decision
```

### 8a — Test via stream_query() (AdkApp native interface)


In [ ]:
import vertexai
from vertexai import agent_engines

vertexai.init(project=PROJECT_ID, location=LOCATION)

planner = agent_engines.get(PLANNER_RESOURCE)

# Create a session
session = planner.create_session(user_id="demo-user")
session_id = session["id"]
print(f"Session created: {session_id}\n")

def run_mission_stream(query: str) -> str:
    """Run a mission query via stream_query() and return the final text response."""
    full_text = ""
    for event in planner.stream_query(
        message=query,
        session_id=session_id,
        user_id="demo-user",
    ):
        if "content" in event and "parts" in event["content"]:
            for part in event["content"]["parts"]:
                if "text" in part:
                    full_text = part["text"]
    return full_text

print("=" * 60)
print("DEMO — Drone Mission Planner via stream_query()")
print("=" * 60)
print()

missions = [
    "Plan a drone delivery mission in Austin, Texas.",
    "Can I fly a delivery drone in Chicago today?",
    "Approve a survey drone flight in Miami, Florida.",
]

for mission in missions:
    print(f"MISSION: {mission}")
    response = run_mission_stream(mission)
    print(f"DECISION:\n{response}")
    print("-" * 60)
    print()


Session created: 451469782286336000

DEMO — Drone Mission Planner via stream_query()

MISSION: Plan a drone delivery mission in Austin, Texas.
DECISION:
Here's the 3-day forecast for Austin, Texas:

*   **Today:** 90°F, Partly Sunny, Wind: 10 to 15 mph S. **CAUTION** — flyable but challenging.
*   **Tonight:** 67°F, Mostly Cloudy, Wind: 5 to 15 mph S. **CAUTION** — flyable but challenging (due to potential for winds up to 15mph).
*   **Saturday:** 74°F, Showers And Thunderstorms Likely, Wind: 5 to 15 mph NNW. **CAUTION** — flyable but challenging, and with showers and thunderstorms likely, conditions could become risky.

**Overall Recommendation:** CAUTION. While flights might be possible, consistent winds in the 10-15 mph range and the potential for thunderstorms on Saturday mean challenging conditions. Drone operators should exercise extreme caution and consider rescheduling if possible.
------------------------------------------------------------

MISSION: Can I fly a delivery drone

### 8b — Test via A2A endpoint (agent-to-agent interop)


### Additional Test Prompts


In [ ]:
# Uncomment to test additional scenarios

# print(run_mission_stream("Aerial photography in Denver, Colorado — safe to fly?"))
# print(run_mission_stream("Medical drone delivery in Seattle, Washington — proceed?"))
# print(run_mission_stream("Deliveries in both Dallas, Texas and Atlanta, Georgia — can I do both?"))
# print(run_mission_stream("Emergency inspection in Phoenix, Arizona — can we go?"))
# print(run_mission_stream("Our drone is rated for 12mph winds — safe in Los Angeles today?"))
# print(run_mission_stream("Full mission briefing for Nashville, Tennessee."))

# Edge case — NWS covers US only
# print(run_mission_stream("Plan a drone mission in London, England."))

print("Uncomment a line above and run.")


---
## Observability — Agent Topology & Logs

### Agent Engine Topology

After running missions, the Agent Engine console shows topology with agent-to-agent edges.  
This requires:
1. **`sub_agents`** (not `AgentTool`) — causes ADK to emit agent-to-agent OTel spans
2. **`opentelemetry-instrumentation-httpx`** installed — makes A2A httpx calls visible to OTel
3. **OTel env vars** set — routes telemetry to the Agent Engine pipeline

Navigate to: **Vertex AI → Agent Engine → [Drone_Mission_Planner_AdkApp_SA] → Traces**

### Agent Identity in Audit Logs

Use Cloud Logging to verify the planner runs as its dedicated SA:


In [ ]:
import subprocess

weather_id = WEATHER_AGENT_RESOURCE.split("/")[-1]
planner_id = PLANNER_RESOURCE.split("/")[-1]

print("=== Weather Agent Logs (last 10) ===")
result = subprocess.run([
    "gcloud", "logging", "read",
    f'resource.type="aiplatform.googleapis.com/ReasoningEngine" '
    f'AND resource.labels.reasoning_engine_id="{weather_id}"',
    "--project", PROJECT_ID, "--limit", "10",
    "--format", "value(timestamp,textPayload)"
], capture_output=True, text=True)
print(result.stdout or "No logs yet — run a mission first.")

print("\n=== Mission Planner Logs (last 10) ===")
result = subprocess.run([
    "gcloud", "logging", "read",
    f'resource.type="aiplatform.googleapis.com/ReasoningEngine" '
    f'AND resource.labels.reasoning_engine_id="{planner_id}"',
    "--project", PROJECT_ID, "--limit", "10",
    "--format", "value(timestamp,textPayload)"
], capture_output=True, text=True)
print(result.stdout or "No logs yet — run a mission first.")

print("\n=== Verify Agent Identity in Audit Logs ===")
print("Use this query in Cloud Logging:")
print(f'  protoPayload.authenticationInfo.principalEmail="drone-mission-planner@{PROJECT_ID}.iam.gserviceaccount.com"')


---
## Reload Deployed Agents After Kernel Restart

Run **Step 1** then this cell to reconnect without redeploying.


In [ ]:
import vertexai
from vertexai import agent_engines

vertexai.init(project=PROJECT_ID, location=LOCATION)

# ── Paste your resource names here ─────────────────────────────────────────
WEATHER_AGENT_RESOURCE = "projects/YOUR_PROJECT_NUMBER/locations/us-central1/reasoningEngines/YOUR_WEATHER_ID"
PLANNER_RESOURCE       = "projects/YOUR_PROJECT_NUMBER/locations/us-central1/reasoningEngines/YOUR_PLANNER_ID"
PLANNER_SA             = f"drone-mission-planner@{PROJECT_ID}.iam.gserviceaccount.com"
# ───────────────────────────────────────────────────────────────────────────

print(f"Weather Agent : {WEATHER_AGENT_RESOURCE}")
print(f"Planner       : {PLANNER_RESOURCE}")
print(f"Agent Identity: {PLANNER_SA}")


Weather Agent : projects/YOUR_PROJECT_NUMBER/locations/us-central1/reasoningEngines/YOUR_WEATHER_ID
Planner       : projects/YOUR_PROJECT_NUMBER/locations/us-central1/reasoningEngines/YOUR_PLANNER_ID
Agent Identity: drone-mission-planner@agentcloud-stable-playground.iam.gserviceaccount.com


---
## Clean Up

Delete agents, the Agent Identity SA, the Maps API key secret, and the staging bucket when done.


In [ ]:
import subprocess

# Uncomment to delete agents
# agent_engines.get(PLANNER_RESOURCE).delete(force=True)
# print(f"Deleted planner: {PLANNER_RESOURCE}")

# agent_engines.get(WEATHER_AGENT_RESOURCE).delete(force=True)
# print(f"Deleted weather agent: {WEATHER_AGENT_RESOURCE}")

# Uncomment to delete the Secret Manager secret
# subprocess.run([
#     "gcloud", "secrets", "delete", "drone-maps-api-key",
#     "--project", PROJECT_ID, "--quiet"
# ], check=True)
# print("Deleted drone-maps-api-key secret")

# Uncomment to delete the Agent Identity SA
# subprocess.run([
#     "gcloud", "iam", "service-accounts", "delete",
#     f"drone-mission-planner@{PROJECT_ID}.iam.gserviceaccount.com",
#     "--project", PROJECT_ID, "--quiet"
# ], check=True)
# print("Deleted Agent Identity SA")

# Uncomment to delete staging bucket (after agents are deleted)
# subprocess.run(["gcloud", "storage", "rm", "--recursive", STAGING_BUCKET], check=True)
# print(f"Deleted bucket: {STAGING_BUCKET}")

print("Uncomment sections above to clean up.")


In [67]:
  import vertexai
  from google.genai import types

  client = vertexai.Client(
      project="agentcloud-stable-playground",
      location="us-central1",
      http_options=types.HttpOptions(
          api_version="v1beta1",
          base_url="https://us-central1-aiplatform.googleapis.com/",
      ),
  )

  agent = client.agent_engines.get(
      name="projects/100161839600/locations/us-central1/reasoningEngines/7384188700504096768"
  )

  session = agent.create_session(user_id="gateway-test")
  print("Session:", session)
  print(dir(agent))
  print(agent.operation_schemas)
  for chunk in agent.stream_query(
      message="Is it safe to fly a drone in Austin TX today?",
      user_id="gateway-test",
      session_id=session["id"],
  ):
      print(chunk)



Session: {'lastUpdateTime': 1776444813.79354, 'appName': '7384188700504096768', 'id': '6895443549890805760', 'state': {}, 'events': [], 'userId': 'gateway-test'}
['__abstractmethods__', '__annotations__', '__class__', '__class_getitem__', '__class_vars__', '__copy__', '__deepcopy__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__fields__', '__fields_set__', '__format__', '__ge__', '__get_pydantic_core_schema__', '__get_pydantic_json_schema__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__pretty__', '__private_attributes__', '__pydantic_complete__', '__pydantic_computed_fields__', '__pydantic_core_schema__', '__pydantic_custom_init__', '__pydantic_decorators__', '__pydantic_extra__', '__pydantic_fields__', '__pydantic_fields_set__', '__pydantic_generic_metadata__', '__pydantic_init_subclass__', '__pydantic_on_complete__', '__pydantic_parent_n

In [70]:
  from google import genai
  from google.genai import types

  client = genai.Client(
      vertexai=True,
      project="agentcloud-stable-playground",
      location="us-central1",
  )

  response = client.models.generate_content(
      model="gemini-2.5-flash",
      contents="Is it safe to fly a drone in Austin TX today?",
      config=types.GenerateContentConfig(
          tools=[types.Tool(
              connected_agents=[types.ConnectedAgent(
                  agent="projects/agentcloud-stable-playground/locations/us-central1/agentGateways/siri-agent-gateway"
              )]
          )]
      )
  )
  print(response.text)

AttributeError: module 'google.genai.types' has no attribute 'ConnectedAgent'

In [74]:
  #!pip install --upgrade google-genai
  import google.genai
  print(google.genai.__version__)

  # Check available tool types
  from google.genai import types
  print([x for x in dir(types) if 'agent' in x.lower() or 'Agent' in x or 'tool' in x.lower() or 'Tool' in x])
  print(types.Tool.model_fields)

1.73.1
['LiveClientToolResponse', 'LiveClientToolResponseDict', 'LiveClientToolResponseOrDict', 'LiveServerToolCall', 'LiveServerToolCallCancellation', 'LiveServerToolCallCancellationDict', 'LiveServerToolCallCancellationOrDict', 'LiveServerToolCallDict', 'LiveServerToolCallOrDict', 'McpCallToolResult', 'Tool', 'ToolCall', 'ToolCallDict', 'ToolCallOrDict', 'ToolCodeExecution', 'ToolCodeExecutionDict', 'ToolCodeExecutionOrDict', 'ToolConfig', 'ToolConfigDict', 'ToolConfigOrDict', 'ToolDict', 'ToolListUnion', 'ToolListUnionDict', 'ToolOrDict', 'ToolParallelAiSearch', 'ToolParallelAiSearchDict', 'ToolParallelAiSearchOrDict', 'ToolResponse', 'ToolResponseDict', 'ToolResponseOrDict', 'ToolType', 'ToolUnion', 'ToolUnionDict']
{'retrieval': FieldInfo(annotation=Union[Retrieval, NoneType], required=False, default=None, alias='retrieval', alias_priority=1, description='Optional. Retrieval tool type. System will always execute the provided retrieval tool(s) to get external knowledge to answer th

In [73]:
  import subprocess, requests

  TOKEN = subprocess.check_output(["gcloud", "auth", "print-access-token"]).decode().strip()

  response = requests.post(
      "https://us-central1-aiplatform.googleapis.com/v1beta1/projects/agentcloud-stable-playground/locations/us-central1/publishers/google/models/gemini-2.5-flash:generateContent",
      headers={
          "Authorization": f"Bearer {TOKEN}",
          "Content-Type": "application/json"
      },
      json={
          "contents": [{"role": "user", "parts": [{"text": "Is it safe to fly a drone in Austin TX today?"}]}],
          "tools": [{
              "connectedAgents": [{
                  "agent": "projects/agentcloud-stable-playground/locations/us-central1/agentGateways/siri-agent-gateway"
              }]
          }]
      }
  )
  print(response.status_code)
  print(response.json())

400
{'error': {'code': 400, 'message': 'Invalid JSON payload received. Unknown name "connectedAgents" at \'tools[0]\': Cannot find field.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.BadRequest', 'fieldViolations': [{'field': 'tools[0]', 'description': 'Invalid JSON payload received. Unknown name "connectedAgents" at \'tools[0]\': Cannot find field.'}]}]}}
